# Initial Data Analysis: Google Year in Search dataset (`trends.csv`)

**Objective.** Load the dataset, inspect its structure with the standard pandas
inspection methods, and record what each output actually says about the data.
The goal at this stage is not to model anything. It is to find out what we have,
what is wrong with it, and what needs cleaning before any real analysis starts.

**Method.** Each section runs one inspection function, shows its output, and is
followed by a short reading of that output. Problems found along the way are
collected into a cleaning checklist at the end.

## 1. Imports and loading

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

print('pandas version:', pd.__version__)

pandas version: 2.2.3


In [2]:
df = pd.read_csv('trends.csv')
print('File loaded successfully.')

File loaded successfully.


The file loads with no parser errors and no `dtype` warnings, which already tells
us something: the column count is consistent on every line, and the quoting is
well formed. That matters here because some queries contain commas and embedded
quotes, such as `Toys "R" Us`. Had the quoting been broken, pandas would have
raised a tokenising error on those rows.

## 2. `shape`: how much data is there?

In [3]:
print('Shape (rows, columns):', df.shape)
print('Number of rows   :', df.shape[0])
print('Number of columns:', df.shape[1])
print('Total cells      :', df.size)

Shape (rows, columns): (26955, 5)
Number of rows   : 26955
Number of columns: 5
Total cells      : 134775


26,955 rows and 5 columns. This is a long, narrow table rather than a wide one,
which is typical of records stored one observation per row.

The row count is worth a second look. 26,955 divided by 5 is 5,391, a whole
number. Every search ranking in this dataset is a top-5 list, so a clean file
should contain a multiple of 5 rows. It does. That is a useful first sanity
check, and section 10 tests the same idea more carefully.

## 3. `head()` and `tail()`: what does a record look like?

In [4]:
df.head(10)

,location,year,category,rank,query
0,Global,2001,Consumer Brands,1,Nokia
1,Global,2001,Consumer Brands,2,Sony
2,Global,2001,Consumer Brands,3,BMW
3,Global,2001,Consumer Brands,4,Palm
4,Global,2001,Consumer Brands,5,Adobe
5,Global,2001,Men,1,Nostradamus
6,Global,2001,Men,2,Osama bin Laden
7,Global,2001,Men,3,Eminem
8,Global,2001,Men,4,Michael Jackson
9,Global,2001,Men,5,Howard Stern


The structure is immediately readable. Each row is one entry in a ranked list:
a **location**, a **year**, a **category**, a **rank** from 1 to 5, and the
**query** that placed at that rank.

The first ten rows are all `Global` / `2001`, moving through `Consumer Brands`
and then `Men`, with rank running 1 to 5 inside each. So the file is sorted by
location, then year, then category, then rank. Rows are grouped into blocks of
five, and each block is one complete top-5 list.

In [5]:
df.tail(10)

,location,year,category,rank,query
26945,Vietnam,2020,Như Thế Nào?,1,Cúng giao thừa như thế nào
26946,Vietnam,2020,Như Thế Nào?,2,Vụ án Hồ Duy Hải như thế nào
26947,Vietnam,2020,Như Thế Nào?,3,Bầu cử tổng thống mỹ như thế nào
26948,Vietnam,2020,Như Thế Nào?,4,Tuấn khỉ bị bắt như thế nào
26949,Vietnam,2020,Như Thế Nào?,5,Gấu đi như thế nào
26950,Vietnam,2020,Là Gì?,1,Virus Corona là gì
26951,Vietnam,2020,Là Gì?,2,Miễn thị thực là gì
26952,Vietnam,2020,Là Gì?,3,Đầu cắt moi là gì
26953,Vietnam,2020,Là Gì?,4,Bệnh bạch hầu là gì
26954,Vietnam,2020,Là Gì?,5,Đông Lào là gì


The last rows are Vietnam, 2020, with categories written in Vietnamese
(`Là Gì?` meaning "what is", `Như Thế Nào?` meaning "how"). This is the first
sign of a problem that section 9 confirms: category labels are not standardised
across countries. They are recorded in each country's own language.

The queries themselves are also non-English and carry full Vietnamese
diacritics, so the file is UTF-8 and pandas has decoded it correctly. Any
downstream text processing has to be Unicode-safe, not ASCII.

## 4. `columns` and `dtypes`: what are the fields?

In [6]:
print('Column names:')
print(list(df.columns))

Column names:
['location', 'year', 'category', 'rank', 'query']


In [7]:
df.dtypes

location    object
year         int64
category    object
rank         int64
query       object
dtype: object

Five columns, and the names are already lowercase with no spaces or trailing
whitespace, so no renaming is needed.

On the types, pandas has inferred:

- `location`, `category`, `query` as `object`, meaning Python strings.
- `year` and `rank` as `int64`.

Both integer columns are correctly typed in the mechanical sense, but neither is
a measurement. `year` is a time label and `rank` is an ordinal position from 1 to
5. Arithmetic on them is meaningless even though pandas will happily perform it.
Section 6 shows exactly how that becomes misleading.

For a cleaned version, `location` and `category` would be better as `category`
dtype: they have very few distinct values relative to 26,955 rows, so the memory
saving is large.

## 5. `info()`: types, non-null counts and memory in one view

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26955 entries, 0 to 26954
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   location  26955 non-null  object
 1   year      26955 non-null  int64 
 2   category  26955 non-null  object
 3   rank      26955 non-null  int64 
 4   query     26955 non-null  object
dtypes: int64(2), object(3)
memory usage: 1.0+ MB


`info()` confirms the shape and types, and adds two things.

First, the non-null count is **26,955 for all five columns**, matching the row
count exactly. There are no missing values anywhere in this file. That is
unusual for real-world data and is explained by how the file was produced: it is
a published, curated ranking rather than raw collected data. Section 7 confirms
this independently.

Second, memory usage is roughly 1 MB. That is small enough that none of the
analysis needs chunking or optimisation.

## 6. `describe()`: summary statistics

In [9]:
df.describe()

,year,rank
count,26955.000000,26955.00000
mean,2015.243369,3.00000
std,3.564683,1.41424
min,2001.000000,1.00000
25%,2013.000000,2.00000
50%,2016.000000,3.00000
75%,2018.000000,4.00000
max,2020.000000,5.00000


By default `describe()` only covers the numeric columns, so we get `year` and
`rank`. Both results need care.

For **`rank`**: mean exactly 3.000000, min 1, max 5, standard deviation
1.41424. These numbers are not findings about the world. Every block contains
exactly one of each rank 1 through 5, so the distribution is uniform by
construction, and the mean of a uniform 1 to 5 is always 3. The standard
deviation of that uniform distribution is sqrt(2) which is 1.41421. Our value
matches to four decimal places. In other words, `rank` behaving this way is
evidence that the file is structurally intact, not evidence about search
behaviour.

For **`year`**: min 2001, max 2020, so the data spans 20 years. The mean of
2015.24 and median of 2016 both sit well above the midpoint of that range, which
means rows are not spread evenly across the years. Recent years contribute many
more rows, because Google expanded Year in Search to more countries and more
categories over time. Any comparison across years has to account for this, since
a later year is represented by more rows simply because more countries were
covered.

The lesson from this cell is that `describe()` ran on two columns where the
statistics are either tautological or misleading. Neither column is a
measurement.

In [10]:
df.describe(include='object')

,location,category,query
count,26955,26955,26955
unique,83,2450,18431
top,United States,People,Paul Walker
freq,2070,760,84


Passing `include='object'` gives the text columns, and this is where the real
information is:

- **`location`**: 83 unique values. The most frequent is `United States` with
  2,070 rows.
- **`category`**: **2,450 unique values**, top is `People` with 760 rows.
- **`query`**: 18,431 unique values, top is `Paul Walker` with 84 rows.

2,450 categories against only 83 locations and 20 years is the single most
important number in this notebook. It is investigated in section 9.

`Paul Walker` appearing 84 times is not an error. He died in November 2013 and
was searched heavily worldwide, so he appears in many countries in the same
year. Repeated queries across locations are expected in this dataset.

### Explicit mean, median, min, max and standard deviation

In [11]:
num = df[['year', 'rank']]

stats = pd.DataFrame({
    'mean'  : num.mean(),
    'median': num.median(),
    'min'   : num.min(),
    'max'   : num.max(),
    'std'   : num.std(),
})
stats

,mean,median,min,max,std
year,2015.243369,2016.0,2001,2020,3.564683
rank,3.000000,3.0,1,5,1.414240


This restates the numbers from `describe()` in the exact form the task asks for.
The interpretation is unchanged: `rank` is uniform by design, and `year` is
skewed towards recent years. The `median` of `year` at 2016 against a range
midpoint of 2010.5 quantifies that skew.

## 7. `isnull().sum()`: missing values

In [12]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total missing values in the dataset:', df.isnull().sum().sum())

Missing values per column:
location    0
year        0
category    0
rank        0
query       0
dtype: int64

Total missing values in the dataset: 0


Zero missing values in every column. Nothing needs imputing or dropping on this
basis.

Since a count of zero can also mean the nulls are hidden rather than absent, the
next cell checks for the usual disguises: empty strings, strings made only of
whitespace, and text placeholders such as `NA`, `null`, `-` or `unknown`.

In [13]:
placeholders = {'', 'na', 'n/a', 'nan', 'null', 'none', '-', '--', 'unknown', '?'}

for col in ['location', 'category', 'query']:
    s = df[col].astype(str)
    blanks = (s.str.strip() == '').sum()
    fake   = s.str.strip().str.lower().isin(placeholders).sum()
    print(f'{col:9s} | empty or whitespace-only: {blanks:4d} | placeholder text: {fake:4d}')

location  | empty or whitespace-only:    0 | placeholder text:    0
category  | empty or whitespace-only:    0 | placeholder text:    0
query     | empty or whitespace-only:    0 | placeholder text:    0


All counts are zero, so the missing values really are absent rather than
disguised. The dataset is genuinely complete on this measure. This is one of the
few places where this file is cleaner than a typical real-world dataset.

## 8. `duplicated().sum()`: duplicate records

In [14]:
print('Number of fully duplicated rows:', df.duplicated().sum())
print('Percentage of the dataset      :', round(100 * df.duplicated().sum() / len(df), 4), '%')

Number of fully duplicated rows: 10
Percentage of the dataset      : 0.0371 %


Ten duplicated rows. That is 0.037 percent of the file, small enough to be easy
to miss and large enough to matter if any count or frequency is computed later.

The count alone does not say whether these are ten scattered accidents or
something systematic, so the next cell prints every duplicated row, including
the first occurrence, using `keep=False`.

In [15]:
dupes = df[df.duplicated(keep=False)].sort_values(['location', 'year', 'category', 'rank'])
dupes

,location,year,category,rank,query
19995,Kazakhstan,2018,Жылдың фильмі/Фильм года,1,Веном
20000,Kazakhstan,2018,Жылдың фильмі/Фильм года,1,Веном
19996,Kazakhstan,2018,Жылдың фильмі/Фильм года,2,Мстители: Война бесконечности
20001,Kazakhstan,2018,Жылдың фильмі/Фильм года,2,Мстители: Война бесконечности
19997,Kazakhstan,2018,Жылдың фильмі/Фильм года,3,Бизнес по-казахски в Америке
20002,Kazakhstan,2018,Жылдың фильмі/Фильм года,3,Бизнес по-казахски в Америке
19998,Kazakhstan,2018,Жылдың фильмі/Фильм года,4,Монстры на каникулах 3
20003,Kazakhstan,2018,Жылдың фильмі/Фильм года,4,Монстры на каникулах 3
19999,Kazakhstan,2018,Жылдың фильмі/Фильм года,5,Дэдпул 2
20004,Kazakhstan,2018,Жылдың фильмі/Фильм года,5,Дэдпул 2


The duplicates are not scattered at all. They form **two complete blocks**, each
repeated exactly once:

1. Kazakhstan, 2018, `Жылдың фильмі/Фильм года` (Kazakh and Russian for "film of
   the year"), ranks 1 to 5.
2. Kenya, 2020, `Trending How To  (Tech)`, ranks 1 to 5.

Each block appears twice in full, giving 20 rows where there should be 10.

This tells us the cause. Ten random duplicate rows would suggest noisy data
entry. Two entire top-5 lists duplicated cleanly suggests the file was assembled
by appending per-country blocks, and two of those blocks were appended twice.
It is an ingestion error, and `drop_duplicates()` removes it safely because the
repeated rows are identical in all five columns.

Note also the category name `Trending How To  (Tech)`, which has a leading space
and a double space in the middle. That is picked up again in section 9.

In [16]:
print('Rows before dropping duplicates:', len(df))
print('Rows after dropping duplicates :', len(df.drop_duplicates()))
print('Rows removed                   :', len(df) - len(df.drop_duplicates()))
print()
print('Is the cleaned row count divisible by 5?', len(df.drop_duplicates()) % 5 == 0)

Rows before dropping duplicates: 26955
Rows after dropping duplicates : 26945


Rows removed                   : 10

Is the cleaned row count divisible by 5? True


26,945 rows after removing the duplicates, and the result is still divisible by
5. The top-5 block structure survives the fix, which is a good sign that we
removed the right rows.

## 9. `nunique()`: unique values and cardinality

In [17]:
print('Unique values per column:')
print(df.nunique())

Unique values per column:


location       83
year           20
category     2450
rank            5
query       18431
dtype: int64


Reading these against the 26,955 total rows:

- **`rank` = 5.** Exactly as expected for top-5 lists.
- **`year` = 20.** A continuous run from 2001 to 2020, confirmed below.
- **`location` = 83.** Countries plus a `Global` aggregate.
- **`query` = 18,431.** High but sensible. Most searches are unique to one
  country and year; the repeats are global events and celebrity deaths.
- **`category` = 2,450.** This one does not fit. With 83 locations and 20 years
  there should be perhaps a few dozen meaningful category types. 2,450 is the
  anomaly of this dataset.

In [18]:
print('Year range:', df.year.min(), 'to', df.year.max())
print('Distinct years:', df.year.nunique())
missing_years = set(range(df.year.min(), df.year.max() + 1)) - set(df.year.unique())
print('Gaps in the year sequence:', missing_years if missing_years else 'none')
print()
print('Rows per year:')
print(df.year.value_counts().sort_index())

Year range: 2001 to 2020
Distinct years: 20
Gaps in the year sequence: none

Rows per year:
year
2001      60
2002     110
2003     155
2004     145
2005      10
2006      75
2007      60
2008     895
2009     445
2010      90
2011     890
2012    2225
2013    2890
2014    2320
2015    2790
2016    3040
2017    2610
2018    2560
2019    2605
2020    2980
Name: count, dtype: int64


The years run 2001 to 2020 with no gaps. The row counts confirm what the mean of
`year` hinted at earlier: the early 2000s contribute only a small number of rows
each, while the 2010s contribute thousands. Coverage grew over time rather than
staying constant, so year-on-year comparisons are comparing different sized
samples.

In [19]:
print('Rank distribution:')
print(df['rank'].value_counts().sort_index())

Rank distribution:
rank
1    5391
2    5391
3    5391
4    5391
5    5391
Name: count, dtype: int64


Each rank appears exactly 5,391 times. Perfectly balanced, which is only
possible if almost every block is a complete 1-to-5 list. Section 10 verifies
that directly.

In [20]:
print('Number of locations:', df.location.nunique())
print()
print('Top 10 locations by row count:')
print(df.location.value_counts().head(10))
print()
print('Bottom 10 locations by row count:')
print(df.location.value_counts().tail(10))

Number of locations: 83

Top 10 locations by row count:
location
United States     2070
Global            1135
Japan              765
Canada             690
Brazil             675
France             630
United Kingdom     590
Finland            555
Mexico             550
Thailand           525
Name: count, dtype: int64

Bottom 10 locations by row count:
location
Zimbabwe              30
Ecuador               20
Myanmar (Burma)       15
Venezuela             15
Sri Lanka             10
Honduras               5
El Salvador            5
Dominican Republic     5
Kuwait                 5
Sudan                  5
Name: count, dtype: int64


`United States` has by far the most rows at 2,070, followed by `Global` at
1,135. The tail has countries with only a handful of rows.

`Global` is worth flagging. It is not a country: it is an aggregate that sits in
the same column as the individual countries. Any analysis that groups by
`location` will double-count unless `Global` is filtered out or handled
separately. This is a classic mixed-granularity problem and it is easy to miss
because nothing in the data marks `Global` as different.

### The category problem

In [21]:
print('Total distinct categories:', df.category.nunique())
print()
print('Top 20 categories by row count:')
print(df.category.value_counts().head(20))

Total distinct categories: 2450

Top 20 categories by row count:
category
People                      760
Searches                    620
Movies                      330
TV Shows                    305
Películas                   250
Songs                       215
Recipes                     175
What is...?                 175
How to...                   170
News                        150
Events                      125
Athletes                    125
Deportistas                 115
Acontecimientos             115
Cómo                        110
Sports                      100
Cómo...                     100
Búsquedas                   100
Fastest Rising Searches     100
Canciones                   100
Name: count, dtype: int64


The top of the list makes the cause obvious. `Movies` appears with 330 rows, and
`Películas`, its Spanish translation, appears separately with 250. `Athletes`
has 125 and `Deportistas` has 115. `How to...` and `Cómo` are the same idea in
two languages.

Category labels were recorded in each country's own language, so one concept is
split across many spellings. The next cell measures how bad this is.

In [22]:
cat_locs  = df.groupby('category')['location'].nunique()
cat_rows  = df.category.value_counts()

print('Categories used in only ONE location :', (cat_locs == 1).sum(), 'of', df.category.nunique())
print('Categories with 5 rows or fewer      :', (cat_rows <= 5).sum(), 'of', df.category.nunique())
print()
print('Share of categories that are single-location:',
      round(100 * (cat_locs == 1).sum() / df.category.nunique(), 1), '%')

Categories used in only ONE location : 2132 of 2450
Categories with 5 rows or fewer      : 1656 of 2450

Share of categories that are single-location: 87.0 %


2,132 of the 2,450 categories appear in only one location, and 1,656 appear in a
single block of five rows. Roughly 87 percent of category labels are used by one
country only.

That is not a category system. It is 83 separate national labelling schemes
stored in one column. Grouping by `category` as it stands would produce over two
thousand groups, most containing five rows, and any cross-country comparison
would be meaningless. This column needs a mapping to a standard English taxonomy
before it can be used, and that mapping is the single largest piece of cleaning
work this dataset requires.

In [23]:
for col in ['location', 'category', 'query']:
    s = df[col].astype(str)
    ws     = (s != s.str.strip()).sum()
    dblsp  = s.str.contains('  ', regex=False).sum()
    casefold_gap = s.nunique() - s.str.lower().nunique()
    print(f'{col:9s} | leading/trailing whitespace: {ws:4d} | internal double spaces: {dblsp:4d} | values differing only by case: {casefold_gap:4d}')

location  | leading/trailing whitespace:    0 | internal double spaces:    0 | values differing only by case:    0
category  | leading/trailing whitespace:  175 | internal double spaces:   30 | values differing only by case:   76


query     | leading/trailing whitespace:    0 | internal double spaces:    4 | values differing only by case:  446


`category` has 175 values carrying leading or trailing whitespace, plus internal
double spaces. `Trending How To  (Tech)` from the duplicate block is one of
them. Values that differ only by capitalisation appear in both `category` and
`query`.

These are cheap to fix with `.str.strip()` and whitespace collapsing, and they
should be fixed first, because they inflate the unique counts before any real
deduplication of meaning is attempted.

## 10. Structural integrity check

The inspection so far suggests the file follows one rule: for every combination
of location, year and category there should be exactly five rows, ranked 1 to 5.
Rather than assume it, this section tests it.

In [24]:
group_sizes = df.groupby(['location', 'year', 'category']).size()

print('Total groups (location, year, category):', len(group_sizes))
print()
print('Distribution of group sizes:')
print(group_sizes.value_counts())
print()
bad = group_sizes[group_sizes != 5]
print('Groups that do NOT have exactly 5 rows:', len(bad))
print(bad)

Total groups (location, year, category): 5389

Distribution of group sizes:
5     5387
10       2
Name: count, dtype: int64

Groups that do NOT have exactly 5 rows: 2
location    year  category                
Kazakhstan  2018  Жылдың фильмі/Фильм года    10
Kenya       2020  Trending How To  (Tech)     10
dtype: int64


5,387 groups contain exactly five rows. Two contain ten. Those two are the
Kazakhstan and Kenya blocks already identified in section 8, so this check finds
the same fault from a completely different direction and finds nothing else.

The rule holds everywhere except the known duplicates. Structurally, the file is
sound.

In [25]:
dup_rank = df.groupby(['location', 'year', 'category'])['rank'].apply(lambda s: s.duplicated().sum())
print('Groups containing a repeated rank:', (dup_rank > 0).sum())

clean = df.drop_duplicates()
sizes_clean = clean.groupby(['location', 'year', 'category']).size()
print('After dropping duplicates, groups not equal to 5:', (sizes_clean != 5).sum())

Groups containing a repeated rank: 2
After dropping duplicates, groups not equal to 5: 0


No group has a repeated rank once the duplicates are removed, and every group is
back to exactly five rows. `drop_duplicates()` fully repairs the structure.

In [26]:
years_per_loc = df.groupby('location')['year'].nunique()

print('Years covered per location:')
print(years_per_loc.describe())
print()
print('Locations present in all 20 years:', (years_per_loc == 20).sum())
print('Locations present in only 1 year :', (years_per_loc == 1).sum())

Years covered per location:
count    83.000000
mean      9.060241
std       4.049539
min       1.000000
25%       6.000000
50%      10.000000
75%      12.000000
max      20.000000
Name: year, dtype: float64

Locations present in all 20 years: 1
Locations present in only 1 year : 8


Coverage is uneven. The median location appears in 10 of the 20 years, some
appear in all 20, and some in only one.

This makes the dataset an **unbalanced panel**. Countries entered and left the
published rankings at different times. Any trend computed across the whole file
will partly reflect which countries happened to be covered in a given year
rather than any change in search behaviour. This is a property of the data, not
an error, but it constrains what can honestly be concluded from it.

## 11. Overall interpretation

### What the dataset is

`trends.csv` holds Google Year in Search rankings: 26,955 rows across five
columns, covering 83 locations and the years 2001 to 2020. Each row is one entry
in a top-5 list, identified by location, year, category and rank. The unit of
observation is a single ranked search query, and the natural key is
`(location, year, category, rank)`.

It is a wide, shallow dataset in the analytical sense. It has excellent breadth
across countries and years but only four descriptive fields per observation, and
no volume, frequency or search-count measure. It records **what** ranked, never
**how much** it ranked by. That limits it to rank-based and frequency-based
analysis; anything requiring magnitude is impossible with this file alone.

### What is in good condition

- No missing values in any column, and none disguised as blanks or placeholder
  text.
- Consistent column count and correct quoting throughout, including queries
  containing commas and embedded quote characters.
- Correct integer types on `year` and `rank`, and correct UTF-8 decoding of
  non-Latin scripts.
- A structural rule of exactly five rows per location-year-category group that
  holds for 5,387 of 5,389 groups.

### What needs cleaning

1. **Category labels, the main problem.** 2,450 distinct values, of which 2,132
   appear in a single location. Labels are recorded in each country's own
   language, so `Movies`, `Películas` and `Filme` are stored as separate
   categories. This column cannot be grouped on until it is mapped to a standard
   taxonomy. This is the largest piece of work.
2. **Ten duplicate rows.** Two complete top-5 blocks (Kazakhstan 2018 and Kenya
   2020) each appear twice, which points to an append error during file
   assembly. `drop_duplicates()` removes them and restores the five-row rule.
3. **Whitespace in category values.** 175 values carry leading or trailing
   spaces, and some contain internal double spaces. Fix with `.str.strip()` and
   whitespace collapsing before any grouping.
4. **`Global` mixed in with countries.** `location` contains both individual
   countries and a `Global` aggregate at a different level of granularity.
   Grouping by location without handling this will double-count.
5. **Misleading numeric columns.** `year` and `rank` are stored as integers but
   neither is a measurement. `rank` is ordinal with a mean fixed at 3.0 by
   construction, and `year` is a time label. `describe()` returns numbers for
   both, and those numbers should not be reported as findings.
6. **Unbalanced coverage across years and locations.** Row counts per year grow
   sharply over the period, and the median location appears in only 10 of the 20
   years. Comparisons across years or countries must account for differing
   coverage rather than treating the panel as complete.

### Suggested cleaning sequence

```text
1. df.drop_duplicates()                          removes the 10 duplicate rows
2. strip and collapse whitespace on text columns  fixes the 175 category values
3. map category to a standard English taxonomy    collapses 2,450 to a usable set
4. separate or flag the Global rows               resolves mixed granularity
5. cast location and category to category dtype   reduces memory
6. treat year and rank as labels, not measures    prevents meaningless statistics
```

### What can be asked of it after cleaning

Once the category column is mapped, the dataset supports questions about which
topics dominate searches in a given year, how quickly a global event propagates
across countries, which queries recur across many locations, and how the mix of
categories shifts over the 20-year span. Any question about search **volume**
needs a different source.